# Stats 1B - Lab Session #04: Matching

This notebook recreates the Lab 04 matching workflow in Python. We use the teaching library `matchingtools` together with `pandas`, `statsmodels`, `scipy`, and plotting libraries to mirror the ideas behind R's `MatchIt`, `optmatch`, and `RItools` workflows.

The empirical question in the lab is whether being a Bolsa Familia beneficiary is associated with voting for Lula in the 2006 Brazilian presidential election. Matching is used as a preprocessing step to make treated and control observations more comparable before estimating treatment effects.

## The Data

The lab dataset contains survey microdata collected in Brazil before the 2006 presidential election.

| Variable | Description |
|---|---|
| `bf` | Bolsa Familia beneficiary indicator (`1=True`, `0=False`) |
| `lula` | Vote for Lula indicator (`1=True`, `0=False`) |
| `gender` | Sex of respondent |
| `age` | Age of respondent |
| `educ` | Education level of respondent, categorical |
| `munsize` | Municipality size, categorical |
| `hdi` | Human development index of municipality |
| `income` | Income level, categorical |
| `region` | Region, categorical |

## Step 0: Load Libraries

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

# Allow importing the local package when this notebook is run from examples/.
PACKAGE_ROOT = Path('..').resolve()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matchingtools as mt

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print('matchingtools version:', mt.__version__)

## Step 1: Load the Data and Inspect It

The path below is relative to this notebook's location in `matchingtools_teaching_v7/examples`.

In [ ]:
DATA_PATH = Path('../../../../data/lab04-DATAlab_atualizado.dta')
d = pd.read_stata(DATA_PATH)

categorical_vars = ['gender', 'educ', 'munsize', 'income', 'region']
for col in categorical_vars:
    if col in d.columns:
        d[col] = d[col].astype('category')

d.head()

In [ ]:
d.info()

Check how many respondents were treated and how many were not.

In [ ]:
d['bf'].value_counts(dropna=False).rename_axis('bf').to_frame('n')

## Step 2: Raw Difference in Means

Before matching, compare the probability of voting for Lula between Bolsa Familia beneficiaries and non-beneficiaries.

In [ ]:
group_means = d.groupby('bf', observed=True)['lula'].agg(['count', 'mean', 'std'])
group_means

In [ ]:
treated_lula = d.loc[d['bf'] == 1, 'lula'].dropna()
control_lula = d.loc[d['bf'] == 0, 'lula'].dropna()

ttest = stats.ttest_ind(treated_lula, control_lula, equal_var=False)
pd.Series({'t_statistic': ttest.statistic, 'p_value': ttest.pvalue})

The same comparison can be written as a regression. The coefficient on `bf` is the raw difference in means.

In [ ]:
ols_raw = smf.ols('lula ~ bf', data=d).fit()
ols_raw.summary().tables[1]

In [ ]:
ols_raw_hc2 = smf.ols('lula ~ bf', data=d).fit(cov_type='HC2')
ols_raw_hc2.summary().tables[1]

## Step 3: Raw Adjusted Regression

For comparison with matching results, estimate an adjusted regression on the raw data using the same variables that will enter the propensity-score model.

In [ ]:
adjusted_formula = 'lula ~ bf + age + C(gender) + hdi + C(region) + C(income) + I(hdi ** 2)'
reg_est = smf.ols(adjusted_formula, data=d).fit(cov_type='HC2')
reg_est.summary().tables[1]

## Step 4: Manual Propensity-Score Matching

The first matching workflow mirrors the R lab's step-by-step approach: estimate a propensity score first, store it in the data, and then match treated and control units using that score.

In [ ]:
covariates = ['age', 'gender', 'hdi', 'region', 'income']
mform = 'bf ~ age + C(gender) + hdi + C(region) + C(income) + I(hdi ** 2)'

ps_logistic = smf.logit(mform, data=d).fit(disp=False)
d['ps'] = ps_logistic.predict(d)

d[['bf', 'ps']].head()

In [ ]:
m = mt.matchit(
    d,
    treatment='bf',
    covariates=covariates,
    method='nearest',
    distance='ps',
    estimand='ATT',
    replace=True,
    ratio=1,
)

print(m)

In [ ]:
dm = mt.match_data(m)

print('Matched data columns:')
print(dm.columns.tolist())

dm['weights'].round(3).value_counts().sort_index().to_frame('n')

Estimate the ATT using the matched data. In the R lab, this was done with `lm_robust`; here we use weighted least squares with HC2 robust standard errors.

In [ ]:
m_est_01 = smf.wls('lula ~ bf', data=dm, weights=dm['weights']).fit(cov_type='HC2')
m_est_01.summary().tables[1]

In [ ]:
m_est_02 = smf.wls(adjusted_formula, data=dm, weights=dm['weights']).fit(cov_type='HC2')
m_est_02.summary().tables[1]

## Step 5: Automatic Matching with `matchingtools`

In practice, you can pass the propensity-score formula directly to `mt.matchit()`. This mirrors the usual `MatchIt` workflow.

In [ ]:
m_alt = mt.matchit(
    d,
    treatment='bf',
    covariates=covariates,
    formula=mform,
    method='nearest',
    estimand='ATT',
    replace=True,
    ratio=1,
)

dm_alt = mt.match_data(m_alt)
print(m_alt)

In [ ]:
# Manual and automatic matching are conceptually equivalent here.
# Small numerical differences can appear because one path uses a stored column
# and the other estimates the propensity score internally.
comparison = pd.DataFrame({
    'manual_matched_n': [len(dm)],
    'automatic_matched_n': [len(dm_alt)],
    'manual_pairs': [len(m.pairs)],
    'automatic_pairs': [len(m_alt.pairs)],
})
comparison

In [ ]:
summary_alt = mt.summary(m_alt, standardize=True)
summary_alt['overview']

In [ ]:
summary_alt['balance']

## Step 6: Balance Diagnostics

Matching is useful only if it improves covariate balance. The cells below reproduce the main MatchIt-style diagnostic workflow in Python.

In [ ]:
fig, ax = mt.love_plot(m_alt)
plt.show()

In [ ]:
mt.plot(m_alt, 'qq', interactive=False)
plt.show()

In [ ]:
mt.plot(m_alt, 'jitter', interactive=False)
plt.show()

In [ ]:
mt.plot(m_alt, 'histogram', interactive=False)
plt.show()

In [ ]:
mt.plot(m_alt, 'density', interactive=False)
plt.show()

### RItools-style balance checks

`matchingtools.xbalance()` is a teaching-oriented analogue of `RItools::xBalance()`. It reports covariate-level mean differences, standardized differences, Welch t-tests, and an overall balance test.

In [ ]:
bal_prematch = mt.xbalance(mform, data=d, report='all', na_rm=False)
print(bal_prematch)
bal_prematch.plot()
plt.show()

In [ ]:
bal_postmatch = mt.xbalance(mform, data=dm_alt, report='all', na_rm=False)
print(bal_postmatch)
bal_postmatch.plot()
plt.show()

## Step 7: Matching Variations

Matching involves many choices. The goal is to find a specification that creates credible balance between treated and control groups.

Notes on differences from the R lab:

- This teaching library supports both `estimand='ATT'` and `estimand='ATC'`.
- Optimal matching supports `ratio > 1` when enough comparison units are available.
- Mahalanobis matching is available through `method='mahalanobis'`.

In [ ]:
# Nearest-neighbor matching with two controls per treated unit.
m2 = mt.matchit(
    d,
    treatment='bf',
    covariates=covariates,
    formula=mform,
    method='nearest',
    replace=True,
    ratio=2,
)
dm2 = mt.match_data(m2)
print(m2)
mt.balance_table(m2)

In [ ]:
# Mahalanobis matching with two controls per treated unit.
m3 = mt.matchit(
    d,
    treatment='bf',
    covariates=covariates,
    method='mahalanobis',
    replace=True,
    ratio=2,
)
dm3 = mt.match_data(m3)
print(m3)
mt.balance_table(m3)

In [ ]:
# Optimal ATC matching with two treated units per control unit.
m4 = mt.matchit(
    d,
    treatment='bf',
    covariates=covariates,
    formula=mform,
    method='optimal',
    estimand='ATC',
    distance='glm',
    ratio=2,
)
dm4 = mt.match_data(m4)
print(m4)
mt.balance_table(m4)

# Your Assignment

For the assignment, use another version of the dataset from Julia Brandao's master's thesis. The data consist of municipality-level observations of standardized test scores (`IDEB`) from 2005, 2007, 2009, and 2011, as well as several other municipality-level variables. Variables whose names start with `D` measure percentage changes.

The substantive goal is to preprocess the data using matching, and then estimate a difference-in-differences model on the balanced dataset to assess the impact of Ceara's change in educational funding rules. The reform was enacted in 2008 only in Ceara and conditioned part of state tax transfers on educational performance indicators.

Write a lab report with no more than 2,000 words, at most two tables and two graphs. You may work with colleagues on the analysis, but each student must submit their own write-up. If you work with others, indicate who you worked with.

## Assignment Tasks

1. Read the assignment data file.
2. Briefly describe the research problem and the data.
3. Report how many municipalities are in the sample and how many were treated.
4. Choose variables to match on and explain your reasoning.
5. Run matching and assess balance; iterate if necessary.
6. Report descriptive results from matching: matched observations, dropped observations, and weights.
7. Use at most one table and one graph to report balance.
8. Reshape the data for difference-in-differences analysis.
9. Estimate a DiD model using four periods with municipality and year fixed effects.
10. Assess the parallel trends assumption with a test statistic.
11. Create one plot that best summarizes the data.
12. Interpret the results.

## Assignment Starter Code

The cells below are scaffolding. They are intentionally not a completed solution.

In [ ]:
ASSIGNMENT_PATH = Path('../../../../data/lab04-DATAassignment.dta')
a = pd.read_stata(ASSIGNMENT_PATH)

a.head()

In [ ]:
# Basic description of the assignment dataset.
display(a.shape)
display(a['CE'].value_counts(dropna=False).rename_axis('CE').to_frame('n'))
a.describe(include='all')

Choose matching covariates. A reasonable starting point is to match on pre-treatment outcomes, pre-treatment trends, and baseline municipality characteristics. Do not match on post-treatment outcomes.

In [ ]:
# Example starter set. Revise this based on your substantive argument.
assignment_covariates = [
    'Dideb0507',
    'Dpop0007',
    'pop2007',
    'idh2000',
    'turmas2007',
    'Dpibpc0007',
    'distcap',
]

assignment_formula = 'CE ~ ' + ' + '.join(assignment_covariates)
assignment_formula

In [ ]:
# Drop rows with missing values in treatment/covariates for the matching step.
match_cols = ['CE'] + assignment_covariates
a_match = a.dropna(subset=match_cols).copy()

ma = mt.matchit(
    a_match,
    treatment='CE',
    covariates=assignment_covariates,
    formula=assignment_formula,
    method='nearest',
    estimand='ATT',
    replace=True,
    ratio=2,
)

dma = mt.match_data(ma)
print(ma)
mt.balance_table(ma)

In [ ]:
# Balance diagnostics for the assignment matching model.
mt.love_plot(ma)
plt.show()

bal_assignment = mt.xbalance(
    data=dma,
    treatment='CE',
    covariates=assignment_covariates,
    report='all',
    na_rm=False,
)
print(bal_assignment)

## Reshape for Difference-in-Differences

The code below sketches the long-format transformation. Check column names in your data before running or adapting this cell.

In [ ]:
id_cols = ['ibgecode', 'mun', 'uf', 'CE', 'weights']
years = [2005, 2007, 2009, 2011]

long_parts = []
for year in years:
    ideb_col = f'ideb{year}'
    pibpc_col = f'pibpc{year}'
    pop_col = f'pop{year}'
    turmas_col = f'turmas{year}'
    available = [col for col in [ideb_col, pibpc_col, pop_col, turmas_col] if col in dma.columns]
    block = dma[id_cols + available].copy()
    block['year'] = year
    rename_map = {
        ideb_col: 'ideb',
        pibpc_col: 'pibpc',
        pop_col: 'pop',
        turmas_col: 'turmas',
    }
    block = block.rename(columns=rename_map)
    long_parts.append(block)

did = pd.concat(long_parts, ignore_index=True)
did['post'] = did['year'].isin([2009, 2011]).astype(int)
did['CE'] = did['CE'].astype(int)
did.head()

In [ ]:
# Four-period DiD with municipality and year fixed effects.
# Clustered standard errors by municipality are a common choice in this setting.
did_model = smf.wls(
    'ideb ~ CE:C(year) + C(ibgecode) + C(year)',
    data=did.dropna(subset=['ideb']),
    weights=did.dropna(subset=['ideb'])['weights'],
).fit(cov_type='cluster', cov_kwds={'groups': did.dropna(subset=['ideb'])['ibgecode']})

did_model.summary().tables[1]

In [ ]:
# A simple trends plot. Improve this for your report as needed.
trend = (
    did.dropna(subset=['ideb'])
    .groupby(['year', 'CE'], as_index=False)
    .apply(lambda g: pd.Series({'ideb_mean': np.average(g['ideb'], weights=g['weights'])}))
    .reset_index(drop=True)
)

sns.lineplot(data=trend, x='year', y='ideb_mean', hue='CE', marker='o')
plt.axvline(2008, linestyle='--', color='black', linewidth=1)
plt.title('Weighted IDEB Trends by Treatment Status')
plt.ylabel('Weighted mean IDEB')
plt.show()

## Interpreting the Assignment Results

Your final write-up should explain why your matching specification is credible, what balance improved after matching, how many observations were retained, and what the DiD estimates imply about the reform. Also discuss whether the pre-treatment trends support the parallel trends assumption.